# Práctica 1 – Ingeniería del Conocimiento: Fase 2: Preprocesado (Parte 2 del cuaderno acumulado)

Este cuaderno continúa la Parte 1 (EDA). Puede pegarse detrás de ella o ejecutarse solo: reutiliza `SEED`, `CHECKS`, `check` y `RUTA_DRIVE` si existen y los define si no. Fija el esquema de validación, mide las hipótesis de preprocesado de la fase 2 y deja preparadas las funciones (`build`, `evaluar`, `resumen`) y las particiones congeladas (`SPLITS`) que usarán las fases siguientes.

| Apartado de la memoria | Sección del cuaderno |
|---|---|
| 2.1 Esquema de validación | F2.1 |
| 2.2 Hipótesis de preprocesado (material, variables, codificación, escalado, cola) | F2.2 |
| 2.3 Diagnóstico del logaritmo en el modelo lineal y redundancias exactas | F2.3 |
| 2.4 Preprocesado recomendado y CV de control | F2.4 |
| Figuras 09 y 10 | F2.5 |
| 2.5 Comprobación de todas las cifras | F2.6 |

**Reglas de la fase.** Todo el preprocesado va dentro de un `Pipeline`/`ColumnTransformer` que se ajusta dentro de cada fold; `id` no es predictora; todas las métricas se calculan en euros; los hiperparámetros son los de por defecto (el ajuste es de la fase 5); una mejora solo cuenta si la diferencia emparejada frente a la referencia supera su desviación entre folds.

In [ ]:
import os
import time
import hashlib
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, KFold

print('pandas', pd.__version__, '| numpy', np.__version__, '| scikit-learn', sklearn.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

# Si este cuaderno se pega detrás de la Parte 1, se reutilizan SEED, CHECKS, check y guardar.
if 'SEED' not in globals():
    SEED = 42
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
os.makedirs('fig_prep', exist_ok=True)

if 'CHECKS' not in globals():
    CHECKS = []

if 'check' not in globals():
    def check(sec, afirmacion, memoria, calculado, dec=2, tol=None):
        """Compara un número de la memoria con el calculado (tolerancia = medio paso del último decimal)."""
        tol = (0.5 * 10 ** -dec + 1e-9) if tol is None else tol
        ok = bool(abs(float(calculado) - float(memoria)) <= tol)
        CHECKS.append((sec, afirmacion, memoria, round(float(calculado), max(dec, 4)), 'OK' if ok else 'REVISAR'))

if 'check_cond' not in globals():
    def check_cond(sec, afirmacion, memoria_txt, condicion, valor):
        """Comprueba una afirmación cualitativa o una desigualdad."""
        CHECKS.append((sec, afirmacion, memoria_txt, valor, 'OK' if bool(condicion) else 'REVISAR'))

def guardar_fig(f, num, nombre, titulo):
    """Guarda la figura en fig_prep/ con un identificador visible y la muestra."""
    f.suptitle(f'Figura {num} · {titulo}  [{num}_{nombre}.png]', fontsize=12, fontweight='bold', x=0.01, ha='left')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f'fig_prep/{num}_{nombre}.png', dpi=110)
    plt.show(); plt.close()

## F2.0 Carga de datos
Igual que en la Parte 1: se leen los CSV de la carpeta de trabajo; si no están, se copian de Drive o se pide subirlos. Las categóricas pasan a `object` (con `NaN` en los faltantes) para que el comportamiento sea el mismo en pandas 2.x y 3.x.

In [ ]:
RUTA_DATOS = ''   # carpeta de los CSV; '' = carpeta de trabajo actual
if 'RUTA_DRIVE' not in globals():   # misma ruta que en la Parte 1 (ajústala si cambia tu carpeta)
    RUTA_DRIVE = '/content/drive/MyDrive/Curso 26-27/Ingeniería del Conocimiento/Prácticas/Práctica 1 - Creación de modelos predictivos de Inteligencia Artificial/Datasets/'

if not (os.path.exists(RUTA_DATOS + 'train-1.csv') and os.path.exists(RUTA_DATOS + 'test-1.csv')):
    # Mismo procedimiento que en la Parte 1: Drive primero y, si falla, subida manual
    try:
        from google.colab import drive
        import shutil
        drive.mount('/content/drive')
        for nombre in ['train-1.csv', 'test-1.csv']:
            shutil.copy(RUTA_DRIVE + nombre, nombre)
        print('CSV copiados desde Drive')
    except Exception as e:
        print('No se pudo copiar desde Drive:', e)
        print('Sube train-1.csv y test-1.csv a mano:')
        from google.colab import files
        files.upload()

tr = pd.read_csv(RUTA_DATOS + 'train-1.csv')
te = pd.read_csv(RUTA_DATOS + 'test-1.csv')

# Variables predictoras (id y price quedan fuera)
CAT = ['category', 'brand', 'brand_tier', 'material', 'season', 'channel', 'country']
NUM = ['weight_kg', 'rating', 'reviews', 'shipping_cost', 'margin', 'discount', 'is_promo']

# Comprobación de que las listas coinciden con los tipos de los datos
feats = tr.drop(columns=['id', 'price'])
assert set(CAT) == set(feats.select_dtypes(exclude='number').columns), 'CAT no coincide con los datos'
assert set(NUM) == set(feats.select_dtypes(include='number').columns), 'NUM no coincide con los datos'

# Las categóricas pasan a dtype object con NaN en los faltantes: mismo comportamiento en pandas 2.x y 3.x
for d in (tr, te):
    for c in CAT:
        d[c] = d[c].astype(object)

X = tr[CAT + NUM].copy()          # sin id ni price
y = tr['price'].to_numpy(dtype=float)
X_test = te[CAT + NUM].copy()     # sin id
print('X:', X.shape, '| X_test:', X_test.shape, '| faltantes de material en train / test:',
      int(X.material.isna().sum()), '/', int(X_test.material.isna().sum()))

## Definiciones: preprocesado, corrección de sesgo del log, validación cruzada y tablas
- `make_prep`: `ColumnTransformer` con las opciones de la fase (tratamiento de `material`, variables excluidas, one-hot o categóricas nativas, escalado).
- `LogSmear`: entrena sobre `log(price)` y devuelve euros; con `correction='duan'` multiplica por el factor de Duan (media de `exp(residuo)`), que corrige el sesgo a la baja de `exp(media del log)`.
- `evaluar`: validación cruzada con particiones dadas; calcula RMSE, MAE, R², RMSE por tramo (≤300 € y >300 €) y el % del error cuadrático que aporta el tramo >300 €.
- `resumen`: tabla con la diferencia emparejada frente a una referencia y el veredicto (MEJORA / EMPATE / EMPEORA).

In [ ]:
UMBRAL = 300   # tramo de precio para el desglose del error: <= 300 € y > 300 €


def make_prep(drop=(), material='desconocido', enc='onehot', scale=False, drop_first=False):
    """Preprocesado dentro de un ColumnTransformer (se ajusta dentro de cada fold).
    drop: variables que se excluyen | material: 'desconocido', 'moda' o 'excluir'
    enc: 'onehot' u 'native' (ordinal + categorical_features de HistGradientBoosting)
    scale: estandariza las numéricas continuas (is_promo, binaria, no se toca)
    drop_first: quita una columna por variable one-hot (evita la colinealidad exacta en el modelo lineal)."""
    cats = [c for c in CAT if c not in drop and c != 'material']
    use_mat = ('material' not in drop) and material != 'excluir'
    cont = [c for c in NUM if c != 'is_promo' and c not in drop]
    binr = [c for c in NUM if c == 'is_promo' and c not in drop]

    def encoder():
        if enc == 'onehot':
            return OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first' if drop_first else None)
        return OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan)

    tf = []
    if use_mat:
        imp = (SimpleImputer(strategy='constant', fill_value='Desconocido') if material == 'desconocido'
               else SimpleImputer(strategy='most_frequent'))
        tf.append(('material', Pipeline([('imp', imp), ('enc', encoder())]), ['material']))
    tf.append(('cat', encoder(), cats))
    tf.append(('num', StandardScaler() if scale else 'passthrough', cont))
    if binr:
        tf.append(('bin', 'passthrough', binr))
    n_cat = (1 if use_mat else 0) + len(cats)
    mask = [True] * n_cat + [False] * (len(cont) + len(binr))   # para categorical_features (modo nativo)
    return ColumnTransformer(tf, remainder='drop'), mask


class LogSmear(RegressorMixin, BaseEstimator):
    """Entrena sobre log(price) y devuelve la predicción en euros.
    correction='none'  : exp(pred)  (sesgo a la baja: es la mediana condicional, no la media)
    correction='duan'  : factor * exp(pred), con factor = media de exp(residuo log) (estimador de Duan)
    residuals='insample': residuos en el propio fold de entrenamiento | 'oof': residuos fuera de muestra
                          (CV interna de n_inner folds, solo con datos de entrenamiento)."""
    def __init__(self, estimator, correction='duan', residuals='insample', n_inner=5, seed=42):
        self.estimator = estimator
        self.correction = correction
        self.residuals = residuals
        self.n_inner = n_inner
        self.seed = seed

    def fit(self, X, y):
        ylog = np.log(np.asarray(y, dtype=float))
        self.est_ = clone(self.estimator).fit(X, ylog)
        if self.correction == 'none':
            self.factor_ = 1.0
        else:
            if self.residuals == 'insample':
                r = ylog - self.est_.predict(X)
            else:
                oof = np.zeros_like(ylog)
                for a, b in KFold(self.n_inner, shuffle=True, random_state=self.seed).split(X):
                    oof[b] = clone(self.estimator).fit(X.iloc[a], ylog[a]).predict(X.iloc[b])
                r = ylog - oof
            self.factor_ = float(np.mean(np.exp(r)))
        return self

    def predict(self, X):
        return self.factor_ * np.exp(self.est_.predict(X))


def build(model, target='direct', **kw):
    """Devuelve el estimador completo (preprocesado + modelo [+ transformación del objetivo])."""
    linear = model in ('lr', 'ridge', 'huber')
    prep, mask = make_prep(drop_first=linear, **kw)
    native = kw.get('enc', 'onehot') == 'native'
    if model == 'lr':
        est = LinearRegression()
    elif model == 'ridge':
        est = Ridge(alpha=1.0)
    elif model == 'huber':
        est = HuberRegressor(max_iter=1000)
    elif model == 'hgb':
        est = HistGradientBoostingRegressor(random_state=SEED, categorical_features=mask if native else None)
    elif model == 'hgb_mae':
        est = HistGradientBoostingRegressor(loss='absolute_error', random_state=SEED)
    elif model == 'gbr':
        est = GradientBoostingRegressor(random_state=SEED)
    elif model == 'gbr_huber':
        est = GradientBoostingRegressor(loss='huber', random_state=SEED)
    else:
        raise ValueError(model)
    pipe = Pipeline([('prep', prep), ('model', est)])
    if target == 'direct':
        return pipe
    if target == 'log_sin':
        return LogSmear(pipe, correction='none')
    if target == 'log_duan':
        return LogSmear(pipe, correction='duan', residuals='insample')
    if target == 'log_duan_oof':
        return LogSmear(pipe, correction='duan', residuals='oof')
    raise ValueError(target)


RES = {}   # resultados por fold de cada variante: RES[nombre] = DataFrame de 15 filas


def evaluar(nombre, est, splits):
    """CV con las particiones dadas. Todas las métricas se calculan en euros (escala original)."""
    t0 = time.time()
    filas = []
    for k, (a, b) in enumerate(splits):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            m = clone(est).fit(X.iloc[a], y[a])
            p = m.predict(X.iloc[b])
        yt = y[b]
        e = p - yt
        sse = float(np.sum(e ** 2))
        lo = yt <= UMBRAL
        hi = ~lo
        filas.append(dict(fold=k,
                          rmse=np.sqrt(sse / len(yt)),
                          mae=float(np.mean(np.abs(e))),
                          r2=1 - sse / float(np.sum((yt - yt.mean()) ** 2)),
                          rmse_le300=np.sqrt(np.mean(e[lo] ** 2)),
                          rmse_gt300=np.sqrt(np.mean(e[hi] ** 2)) if hi.any() else np.nan,
                          share_gt300=float(np.sum(e[hi] ** 2)) / sse))
    df = pd.DataFrame(filas)
    df.attrs['seg'] = time.time() - t0
    RES[nombre] = df
    return df


def resumen(nombres, ref):
    """Tabla: media y desviación del RMSE (15 folds), diferencia emparejada frente a `ref`,
    RMSE por tramo (<= 300 € y > 300 €), % del error cuadrático que aporta el tramo > 300 €, MAE y R².
    Una MEJORA solo cuenta si |media de la diferencia| > su desviación entre folds (regla del enunciado
    del proyecto). EMPEORA es descriptivo: peor por encima del ruido o peor en los 15 folds."""
    filas = []
    r = RES[ref].rmse.to_numpy()
    for n in nombres:
        d = RES[n]
        dif = d.rmse.to_numpy() - r
        m, s = dif.mean(), dif.std(ddof=1)
        if n == ref:
            ver = '(referencia)'
        elif abs(m) < 1e-6:
            ver = 'IDÉNTICO'
        elif m < 0 and abs(m) > s:
            ver = 'MEJORA'
        elif (m > 0 and m > s) or (dif >= 0).all():
            ver = 'EMPEORA'    # peor por encima del ruido, o peor en los 15 folds (solo descriptivo)
        else:
            ver = 'EMPATE'
        filas.append(dict(variante=n, RMSE=d.rmse.mean(), std=d.rmse.std(ddof=1), dif_media=m, dif_std=s,
                          folds_mejor=int((dif < 0).sum()), RMSE_le300=d.rmse_le300.mean(),
                          RMSE_gt300=d.rmse_gt300.mean(), pct_MSE_gt300=100 * d.share_gt300.mean(),
                          MAE=d.mae.mean(), R2=d.r2.mean(), veredicto=ver))
    return pd.DataFrame(filas).set_index('variante')


def mostrar(df):
    print(df.round({'RMSE': 3, 'std': 3, 'dif_media': 3, 'dif_std': 3, 'RMSE_le300': 3, 'RMSE_gt300': 3,
                    'pct_MSE_gt300': 1, 'MAE': 3, 'R2': 4}).to_string())


def firma(splits):
    """Huella de las particiones (para comprobar que son idénticas en otro entorno)."""
    return hashlib.md5(np.concatenate([b for _, b in splits]).astype(np.int64).tobytes()).hexdigest()[:12]

## F2.1 Esquema de validación (memoria 2.1)
Se compara una sola vez, sin ajustar nada más, la CV repetida 5x3 con KFold aleatorio frente a folds estratificados por tramos de precio, con la regresión lineal y HistGradientBoosting por defecto. **Criterio fijado antes de ver los resultados:** estratificado solo si reduce al menos un 10 % la desviación entre folds del RMSE de ambos modelos y la de la diferencia emparejada lineal − HGB (que es el ruido que afecta a toda comparación posterior); si no, KFold aleatorio, que es el más simple. El esquema elegido queda congelado en `SPLITS`.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 1. ESQUEMA DE VALIDACIÓN
# Objetivo: medir cada modelo siempre con las MISMAS particiones, para que las diferencias entre
# modelos sean del modelo y no del azar del reparto.
# Se compara una sola vez dos formas de repartir los datos (aleatoria y estratificada por precio)
# y se aplica una regla fijada ANTES de ver resultados (ver paso 4).
# ---------------------------------------------------------------------------------------------

# PASO 1. Dos formas de repartir las 8.000 filas: 5 folds x 3 repeticiones = 15 particiones.
# Cada partición es un par (filas de entrenamiento, filas de validación).
SPLITS_ALEATORIO = list(RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(X))

# Estratificado: se corta el precio en 8 tramos (más finos en la cola cara) y cada fold recibe
# la misma proporción de cada tramo. Las etiquetas solo sirven para repartir, no son una variable.
tramos = pd.qcut(y, q=[0, .2, .4, .6, .8, .9, .95, .99, 1.0], labels=False)
SPLITS_ESTRAT = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(X, tramos))
print('filas por tramo de precio:', np.bincount(tramos).tolist())

# PASO 2. ¿Cuántos productos de más de 300 € cae en cada fold de validación?
n_caros_aleatorio, n_caros_estrat = [], []
for _, val in SPLITS_ALEATORIO:
    n_caros_aleatorio.append(int((y[val] > UMBRAL).sum()))
for _, val in SPLITS_ESTRAT:
    n_caros_estrat.append(int((y[val] > UMBRAL).sum()))
bal = pd.DataFrame({'aleatorio': n_caros_aleatorio, 'estratificado': n_caros_estrat})
print('\nProductos > 300 € por fold de validación (15 folds):')
print(bal.agg(['min', 'max', 'mean', 'std']).round(2).T)

# PASO 3. Medir los dos modelos de referencia (valores por defecto) con cada reparto.
REF_LIN = dict(model='lr', scale=True)    # regresión lineal
REF_HGB = dict(model='hgb')               # HistGradientBoosting
for nombre, particiones in [('aleatorio', SPLITS_ALEATORIO), ('estratificado', SPLITS_ESTRAT)]:
    evaluar(f'ESQ-{nombre}|LIN', build(**REF_LIN), particiones)
    evaluar(f'ESQ-{nombre}|HGB', build(**REF_HGB), particiones)

# Resumen: media y desviación del RMSE de los 15 folds, y de la DIFERENCIA lineal - HGB fold a fold.
filas = []
for nombre in ['aleatorio', 'estratificado']:
    rmse_lin = RES[f'ESQ-{nombre}|LIN'].rmse.to_numpy()
    rmse_hgb = RES[f'ESQ-{nombre}|HGB'].rmse.to_numpy()
    dif = rmse_lin - rmse_hgb
    filas.append(dict(esquema=nombre,
                      RMSE_LIN=rmse_lin.mean(), std_LIN=rmse_lin.std(ddof=1),
                      RMSE_HGB=rmse_hgb.mean(), std_HGB=rmse_hgb.std(ddof=1),
                      dif_LIN_menos_HGB=dif.mean(), std_dif=dif.std(ddof=1)))
esq_tab = pd.DataFrame(filas).set_index('esquema')
print('\n', esq_tab.round(3).to_string())

# PASO 4. Regla (fijada antes de mirar): elegir el estratificado solo si baja al menos un 10 %
# la desviación de las TRES magnitudes. La tercera (la de la diferencia entre modelos) es la que
# importa, porque es el ruido que afecta a toda comparación posterior.
def reduccion(col):
    """Cuánto baja la desviación con folds estratificados (positivo = varía menos)."""
    return 1 - esq_tab.loc['estratificado', col] / esq_tab.loc['aleatorio', col]

red = {'std RMSE lineal': reduccion('std_LIN'),
       'std RMSE HGB': reduccion('std_HGB'),
       'std diferencia lineal-HGB': reduccion('std_dif')}
print('\nReducción de la desviación con folds estratificados:')
for nombre, valor in red.items():
    print(f'  {nombre}: {valor * 100:+.1f} %')

if min(red.values()) >= 0.10:
    ESQUEMA, SPLITS = 'estratificado', SPLITS_ESTRAT
else:
    ESQUEMA, SPLITS = 'aleatorio', SPLITS_ALEATORIO
print(f'\n>>> ESQUEMA ELEGIDO (congelado para todo lo que sigue): {ESQUEMA}')

# Las referencias de las fases siguientes son estas mismas ejecuciones (mismas particiones).
RES['LIN|ref'] = RES[f'ESQ-{ESQUEMA}|LIN']
RES['HGB|ref'] = RES[f'ESQ-{ESQUEMA}|HGB']

## F2.2 Hipótesis de preprocesado (memoria 2.2)
Cada variante cambia **un solo factor** respecto a la referencia, con las particiones congeladas:
1. `material`: categoría «Desconocido» (referencia) / imputación por moda / excluir la variable.
2. Con y sin `brand_tier`, `is_promo` y `country`.
3. Codificación en HistGradientBoosting: one-hot (referencia) / categóricas nativas.
4. Escalado en el modelo lineal (mínimos cuadrados y, como extra, Ridge con `alpha=1`).
5. Cola del precio: objetivo directo / log sin corregir / log con corrección de Duan / pérdidas robustas (`HuberRegressor`, `absolute_error` en HGB y Huber frente a cuadrática en `GradientBoostingRegressor`, porque HGB no tiene Huber).

Transformar variables (log de `weight_kg`, ratios, interacciones) queda para la fase 4. Tarda unos 2 minutos.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 2. HIPÓTESIS DE LA FASE 2, una a una frente a la referencia (mismas particiones congeladas)
# Diseño "un factor cada vez": cada variante cambia SOLO una cosa respecto a la referencia.
# ---------------------------------------------------------------------------------------------
t_ini = time.time()

# --- Modelo lineal -------------------------------------------------------------------------------
LIN = {
    'LIN|material=moda':        dict(model='lr', scale=True, material='moda'),
    'LIN|material=excluir':     dict(model='lr', scale=True, material='excluir'),
    'LIN|sin brand_tier':       dict(model='lr', scale=True, drop=('brand_tier',)),
    'LIN|sin is_promo':         dict(model='lr', scale=True, drop=('is_promo',)),
    'LIN|sin country':          dict(model='lr', scale=True, drop=('country',)),
    'LIN|sin escalar':          dict(model='lr', scale=False),
    'LIN|log sin corregir':     dict(model='lr', scale=True, target='log_sin'),
    'LIN|log + Duan':           dict(model='lr', scale=True, target='log_duan'),
    'LIN|Huber':                dict(model='huber', scale=True),
}
# Extra para la hipótesis 4: en mínimos cuadrados el escalado no cambia las predicciones; con
# penalización (Ridge, alpha=1 por defecto, sin ajustar) sí puede cambiarlas.
LIN_RIDGE = {
    'RIDGE|escalado':           dict(model='ridge', scale=True),
    'RIDGE|sin escalar':        dict(model='ridge', scale=False),
}
for n, kw in {**LIN, **LIN_RIDGE}.items():
    evaluar(n, build(**kw), SPLITS)
print(f'[lineal listo en {time.time() - t_ini:.0f} s]')

# --- HistGradientBoosting ------------------------------------------------------------------------
HGB = {
    'HGB|material=moda':        dict(model='hgb', material='moda'),
    'HGB|material=excluir':     dict(model='hgb', material='excluir'),
    'HGB|sin brand_tier':       dict(model='hgb', drop=('brand_tier',)),
    'HGB|sin is_promo':         dict(model='hgb', drop=('is_promo',)),
    'HGB|sin country':          dict(model='hgb', drop=('country',)),
    'HGB|categóricas nativas':  dict(model='hgb', enc='native'),
    'HGB|log sin corregir':     dict(model='hgb', target='log_sin'),
    'HGB|log + Duan (train)':   dict(model='hgb', target='log_duan'),
    'HGB|log + Duan (OOF)':     dict(model='hgb', target='log_duan_oof'),
    'HGB|absolute_error':       dict(model='hgb_mae'),
}
# Pérdida Huber: HistGradientBoosting no la tiene; se compara en GradientBoostingRegressor
GBR = {
    'GBR|squared_error':        dict(model='gbr'),
    'GBR|huber':                dict(model='gbr_huber'),
}
for n, kw in {**HGB, **GBR}.items():
    evaluar(n, build(**kw), SPLITS)
print(f'[HGB y GBR listos en {time.time() - t_ini:.0f} s]')

# --- Tablas --------------------------------------------------------------------------------------
H1 = ['material=moda', 'material=excluir']
H2 = ['sin brand_tier', 'sin is_promo', 'sin country']

print('\n===== MODELO LINEAL (referencia: regresión lineal, Desconocido, todas las variables, one-hot drop=first, escalado) =====')
print('\n-- Hipótesis 1 y 2: material y variables redundantes/débiles')
tab_lin_12 = resumen(['LIN|ref'] + ['LIN|' + h for h in H1 + H2], 'LIN|ref'); mostrar(tab_lin_12)
print('\n-- Hipótesis 4: escalado (mínimos cuadrados y Ridge)')
tab_lin_4 = resumen(['LIN|ref', 'LIN|sin escalar'], 'LIN|ref'); mostrar(tab_lin_4)
tab_ridge = resumen(['RIDGE|escalado', 'RIDGE|sin escalar'], 'RIDGE|escalado'); mostrar(tab_ridge)
print('\n-- Hipótesis 5: cola (objetivo)')
tab_lin_5 = resumen(['LIN|ref', 'LIN|log sin corregir', 'LIN|log + Duan', 'LIN|Huber'], 'LIN|ref'); mostrar(tab_lin_5)

print('\n===== HistGradientBoosting (referencia: por defecto, Desconocido, todas las variables, one-hot) =====')
print('\n-- Hipótesis 1, 2 y 3')
tab_hgb_123 = resumen(['HGB|ref'] + ['HGB|' + h for h in H1 + H2 + ['categóricas nativas']], 'HGB|ref'); mostrar(tab_hgb_123)
print('\n-- Hipótesis 5: cola (objetivo)')
tab_hgb_5 = resumen(['HGB|ref', 'HGB|log sin corregir', 'HGB|log + Duan (train)', 'HGB|log + Duan (OOF)',
                     'HGB|absolute_error'], 'HGB|ref'); mostrar(tab_hgb_5)
print('\n-- Pérdida Huber frente a cuadrática en GradientBoostingRegressor (referencia propia: GBR squared_error)')
tab_gbr = resumen(['GBR|squared_error', 'GBR|huber'], 'GBR|squared_error'); mostrar(tab_gbr)
print(f'\nTiempo total de esta sección: {time.time() - t_ini:.0f} s')

## F2.3 Diagnóstico: ¿por qué el log hunde al modelo lineal? (memoria 2.3)
Con el peso sin transformar, el modelo lineal sobre `log(price)` predice `exp(coef · peso)`: los pocos productos muy pesados reciben precios absurdos al volver a euros. La transformación del peso es de la fase 4; aquí solo se documenta el fallo.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 3. DIAGNÓSTICO: ¿por qué el log sin más hunde al modelo lineal (RMSE ~100 € frente a ~54 €)?
# En un modelo lineal sobre log(price) el peso entra como exp(coef * peso): unos pocos productos muy
# pesados reciben precios absurdos al volver a euros. Se mira la predicción máxima de cada fold.
# (La transformación de weight_kg es de la Fase 4; aquí solo se explica el fallo.)
# ---------------------------------------------------------------------------------------------
est_dbg = build('lr', scale=True, target='log_sin')
filas = []
for k, (a, b) in enumerate(SPLITS):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m = clone(est_dbg).fit(X.iloc[a], y[a])
        p = m.predict(X.iloc[b])
    i = int(np.argmax(p))
    filas.append(dict(fold=k, RMSE=RES['LIN|log sin corregir'].rmse[k], pred_max=p[i],
                      peso_kg=X.iloc[b].weight_kg.values[i], precio_real=y[b][i]))
dbg = pd.DataFrame(filas)
print(dbg.round(1).to_string(index=False))
print('\nFolds con RMSE > 100 €:', int((dbg.RMSE > 100).sum()), 'de 15 | todos contienen un producto con peso >= 14 kg:',
      bool((dbg.loc[dbg.RMSE > 100, 'peso_kg'] >= 14).all()))
print('Productos con weight_kg > 10: train', int((X.weight_kg > 10).sum()), '| test', int((X_test.weight_kg > 10).sum()),
      '| peso máximo train / test:', X.weight_kg.max(), '/', X_test.weight_kg.max())

## F2.4 Preprocesado recomendado y CV de control (memoria 2.4)
Las decisiones que salieron «empate» se resuelven con la opción más simple (excluir `material`, `brand_tier`, `is_promo` y `country`) y se mide el conjunto entero frente a la referencia. Después se repite lo esencial con **otra semilla de CV** (2026). Esa CV de control no se usa para elegir nada: solo confirma que lo elegido no depende de las particiones.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 4. PREPROCESADO RECOMENDADO: se aplican a la vez las decisiones que salieron "empate" (se elige la
# opción más simple) y se mide el conjunto frente a la referencia. Después, una CV de CONTROL con otra
# semilla (no se usa para elegir nada, solo para confirmar que lo elegido no depende de las particiones).
# ---------------------------------------------------------------------------------------------
SIN = ('material', 'brand_tier', 'is_promo', 'country')   # variables que se excluyen en el conjunto reducido

COMBO = {
    'LIN|reducido':                  dict(model='lr', scale=True, drop=SIN),
    'HGB|reducido':                  dict(model='hgb', drop=SIN),
    'HGB|reducido + log Duan (train)': dict(model='hgb', drop=SIN, target='log_duan'),
}
for n, kw in COMBO.items():
    evaluar(n, build(**kw), SPLITS)

print('===== Conjunto reducido (sin material, brand_tier, is_promo, country) frente a la referencia, CV congelada =====')
print('\n-- Lineal'); tab_combo_lin = resumen(['LIN|ref', 'LIN|reducido'], 'LIN|ref'); mostrar(tab_combo_lin)
print('\n-- HistGradientBoosting')
tab_combo_hgb = resumen(['HGB|ref', 'HGB|reducido', 'HGB|log + Duan (train)', 'HGB|reducido + log Duan (train)'], 'HGB|ref')
mostrar(tab_combo_hgb)

# --- CV de control (otra semilla) ----------------------------------------------------------------
SEED_CONTROL = 2026
SPLITS_CTRL = list(RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED_CONTROL).split(X))
print(f'\nCV de control: semilla {SEED_CONTROL} (la CV congelada usa la semilla {SEED})')
CTRL = {
    'C|LIN|ref':                       dict(model='lr', scale=True),
    'C|LIN|reducido':                  dict(model='lr', scale=True, drop=SIN),
    'C|HGB|ref':                       dict(model='hgb'),
    'C|HGB|reducido':                  dict(model='hgb', drop=SIN),
    'C|HGB|log + Duan (train)':        dict(model='hgb', target='log_duan'),
    'C|HGB|reducido + log Duan (train)': dict(model='hgb', drop=SIN, target='log_duan'),
}
for n, kw in CTRL.items():
    evaluar(n, build(**kw), SPLITS_CTRL)
print('\n-- Control, lineal'); tab_ctrl_lin = resumen(['C|LIN|ref', 'C|LIN|reducido'], 'C|LIN|ref'); mostrar(tab_ctrl_lin)
print('\n-- Control, HistGradientBoosting')
tab_ctrl_hgb = resumen(['C|HGB|ref', 'C|HGB|reducido', 'C|HGB|log + Duan (train)', 'C|HGB|reducido + log Duan (train)'], 'C|HGB|ref')
mostrar(tab_ctrl_hgb)

## F2.5 Figuras (memoria 2.1 y 2.2)
Figura 09: esquema de validación. Figura 10: diferencias emparejadas de cada variante frente a su referencia (el modelo lineal con log se omite porque su diferencia, +46 €, rompe la escala; está en la tabla de F2.2).

In [ ]:
# Figura 09: esquema de validación (aleatorio vs estratificado)
f, ax = plt.subplots(1, 2, figsize=(13, 4.5))
datos, etiq = [], []
for modelo in ['LIN', 'HGB']:
    for esq in ['aleatorio', 'estratificado']:
        datos.append(RES[f'ESQ-{esq}|{modelo}'].rmse.values)
        etiq.append(f'{"Lineal" if modelo == "LIN" else "HGB"}\n{esq}')
ax[0].boxplot(datos, showmeans=True)
ax[0].set_xticks(range(1, 5)); ax[0].set_xticklabels(etiq)
for i, d in enumerate(datos, start=1):
    ax[0].scatter(np.full(len(d), i) + np.linspace(-.12, .12, len(d)), d, s=10, alpha=.5, c='k')
ax[0].set_ylabel('RMSE por fold (€)'); ax[0].set_title('RMSE en los 15 folds'); ax[0].grid(alpha=.3)
ax[1].boxplot([bal.aleatorio, bal.estratificado], showmeans=True)
ax[1].set_xticks([1, 2]); ax[1].set_xticklabels(['aleatorio', 'estratificado'])
ax[1].set_ylabel('productos > 300 € en el fold de validación'); ax[1].set_title('Reparto de la cola entre folds'); ax[1].grid(alpha=.3)
guardar_fig(f, '09', 'esquema_cv', 'Validación: KFold aleatorio frente a folds estratificados por precio')

# Figura 10: diferencia emparejada (media ± desviación entre folds) de cada variante frente a su referencia
vars_fig = [
    ('Lineal', 'LIN|ref', ['LIN|material=moda', 'LIN|material=excluir', 'LIN|sin brand_tier', 'LIN|sin is_promo',
                           'LIN|sin country', 'LIN|sin escalar', 'LIN|Huber']),
    ('HGB', 'HGB|ref', ['HGB|material=moda', 'HGB|material=excluir', 'HGB|sin brand_tier', 'HGB|sin is_promo',
                        'HGB|sin country', 'HGB|categóricas nativas', 'HGB|log sin corregir', 'HGB|log + Duan (train)',
                        'HGB|log + Duan (OOF)', 'HGB|absolute_error']),
]
f, ax = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
for a_, (tit, ref, lista) in zip(ax, vars_fig):
    mm, ss, nn = [], [], []
    for n in lista:
        dif = RES[n].rmse.values - RES[ref].rmse.values
        mm.append(dif.mean()); ss.append(dif.std(ddof=1)); nn.append(n.split('|')[1])
    pos = np.arange(len(lista))[::-1]
    col = ['#2f855a' if (m < 0 and abs(m) > s) else ('#c53030' if (m > 0 and m > s) else '#718096') for m, s in zip(mm, ss)]
    a_.errorbar(mm, pos, xerr=ss, fmt='none', ecolor='#a0aec0', capsize=3)
    a_.scatter(mm, pos, c=col, s=45, zorder=3)
    a_.axvline(0, c='k', lw=1); a_.set_yticks(pos); a_.set_yticklabels(nn)
    a_.set_title(f'{tit}: Δ RMSE frente a la referencia (barra = ± desviación entre folds)'); a_.set_xlabel('Δ RMSE (€); negativo = mejor')
    a_.grid(alpha=.3)
ax[0].set_xlim(-1.6, 1.6)
guardar_fig(f, '10', 'hipotesis_dif', 'Diferencias emparejadas por variante (verde = mejora real, rojo = empeora, gris = empate)')

## F2.6 Comprobación de cifras (memoria 2.5)
Cada cifra citada en la memoria se compara con la calculada en este cuaderno. Las tolerancias son de 0,05 € en RMSE, MAE y diferencias, y de ±1 fold en el recuento de folds mejores. Un `REVISAR` indica una diferencia entre entornos (versiones de librerías), no necesariamente un error.

In [ ]:
# ---------------------------------------------------------------------------------------------
# 6. COMPROBACIÓN DE CIFRAS: lo que dice la memoria (valores copiados de la ejecución en Colab con
# pandas 2.2.3, numpy 2.4.6 y scikit-learn 1.6.1) frente a lo que calcula este cuaderno ahora.
# Si una fila sale REVISAR, cambia la cifra de la memoria o investiga la diferencia (versiones de librerías).
# ---------------------------------------------------------------------------------------------
# nombre: (RMSE medio, std RMSE, dif. media emparejada, std dif., folds mejores de 15, RMSE<=300, RMSE>300, % MSE>300, MAE, R2, veredicto)
MEMORIA = {
    'LIN|ref': (54.063, 5.034, 0.0, 0.0, 0, 38.802, 173.403, 49.8, 34.803, 0.5256, '(referencia)'),
    'HGB|ref': (54.625, 4.591, 0.0, 0.0, 0, 40.999, 166.923, 45.3, 35.431, 0.515, '(referencia)'),
    'LIN|material=moda': (54.059, 5.035, -0.004, 0.005, 13, 38.798, 173.396, 49.9, 34.801, 0.5256, 'EMPATE'),
    'LIN|material=excluir': (54.063, 5.053, 0.0, 0.053, 8, 38.763, 173.586, 49.9, 34.794, 0.5256, 'EMPATE'),
    'LIN|sin brand_tier': (54.063, 5.034, 0.0, 0.0, 2, 38.802, 173.403, 49.8, 34.803, 0.5256, 'IDÉNTICO'),
    'LIN|sin is_promo': (54.057, 5.033, -0.006, 0.007, 12, 38.795, 173.394, 49.9, 34.797, 0.5257, 'EMPATE'),
    'LIN|sin country': (54.059, 5.027, -0.004, 0.038, 8, 38.79, 173.429, 49.9, 34.797, 0.5256, 'EMPATE'),
    'LIN|sin escalar': (54.063, 5.034, 0.0, 0.0, 4, 38.802, 173.403, 49.8, 34.803, 0.5256, 'IDÉNTICO'),
    'LIN|log sin corregir': (100.008, 53.636, 45.945, 52.798, 0, 40.098, 399.134, 71.4, 36.51, -1.009, 'EMPEORA'),
    'LIN|log + Duan': (102.707, 56.731, 48.644, 55.92, 0, 40.756, 409.64, 70.8, 36.973, -1.1464, 'EMPEORA'),
    'LIN|Huber': (54.523, 5.297, 0.46, 0.386, 2, 37.561, 181.67, 53.7, 34.076, 0.5177, 'EMPEORA'),
    'RIDGE|escalado': (54.062, 5.035, 0.0, 0.0, 0, 38.798, 173.418, 49.9, 34.801, 0.5256, '(referencia)'),
    'RIDGE|sin escalar': (54.064, 5.046, 0.002, 0.019, 8, 38.756, 173.615, 50.0, 34.774, 0.5255, 'EMPATE'),
    'HGB|material=moda': (54.606, 4.637, -0.018, 0.326, 9, 40.949, 167.122, 45.4, 35.415, 0.5153, 'EMPATE'),
    'HGB|material=excluir': (54.696, 4.762, 0.072, 0.319, 4, 41.076, 167.084, 45.2, 35.475, 0.5137, 'EMPATE'),
    'HGB|sin brand_tier': (54.575, 4.638, -0.05, 0.334, 9, 40.998, 166.641, 45.3, 35.417, 0.516, 'EMPATE'),
    'HGB|sin is_promo': (54.624, 4.591, -0.001, 0.004, 1, 40.997, 166.925, 45.3, 35.43, 0.515, 'EMPATE'),
    'HGB|sin country': (54.604, 4.578, -0.021, 0.262, 7, 41.032, 166.702, 45.2, 35.431, 0.5154, 'EMPATE'),
    'HGB|categóricas nativas': (54.358, 4.546, -0.267, 0.434, 9, 40.922, 165.594, 45.0, 35.256, 0.5198, 'EMPATE'),
    'HGB|log sin corregir': (54.274, 4.901, -0.351, 0.826, 9, 39.031, 173.947, 49.8, 34.379, 0.5218, 'EMPATE'),
    'HGB|log + Duan (train)': (53.985, 4.829, -0.639, 0.734, 12, 39.43, 170.26, 48.2, 34.569, 0.5268, 'EMPATE'),
    'HGB|log + Duan (OOF)': (53.925, 4.755, -0.7, 0.668, 13, 39.964, 167.362, 46.7, 34.881, 0.5277, 'MEJORA'),
    'HGB|absolute_error': (54.999, 5.193, 0.374, 1.31, 6, 38.109, 182.269, 53.2, 34.357, 0.5095, 'EMPATE'),
    'GBR|squared_error': (54.148, 5.227, 0.0, 0.0, 0, 39.407, 171.093, 48.4, 34.749, 0.5238, '(referencia)'),
    'GBR|huber': (54.502, 5.155, 0.353, 0.789, 6, 37.9, 180.046, 52.8, 34.252, 0.5181, 'EMPATE'),
    'LIN|reducido': (54.054, 5.046, -0.009, 0.06, 9, 38.744, 173.603, 50.0, 34.784, 0.5257, 'EMPATE'),
    'HGB|reducido': (54.698, 4.676, 0.073, 0.391, 4, 41.055, 167.129, 45.3, 35.46, 0.5139, 'EMPATE'),
    'HGB|reducido + log Duan (train)': (54.131, 4.857, -0.494, 0.809, 11, 39.596, 170.419, 48.0, 34.649, 0.5243, 'EMPATE'),
    'C|LIN|ref': (53.969, 6.406, 0.0, 0.0, 0, 38.793, 171.277, 48.9, 34.786, 0.5267, '(referencia)'),
    'C|LIN|reducido': (53.987, 6.408, 0.018, 0.073, 6, 38.762, 171.542, 49.0, 34.791, 0.5264, 'EMPATE'),
    'C|HGB|ref': (54.432, 5.793, 0.0, 0.0, 0, 41.166, 162.827, 43.7, 35.389, 0.5173, '(referencia)'),
    'C|HGB|reducido': (54.49, 5.847, 0.058, 0.376, 5, 41.118, 163.504, 43.9, 35.402, 0.5166, 'EMPATE'),
    'C|HGB|log + Duan (train)': (53.826, 6.145, -0.606, 0.947, 11, 39.543, 166.659, 46.7, 34.608, 0.5291, 'EMPATE'),
    'C|HGB|reducido + log Duan (train)': (53.921, 6.302, -0.511, 0.865, 9, 39.569, 167.209, 46.8, 34.666, 0.5274, 'EMPATE'),
}
REF_DE = {n: ('C|LIN|ref' if n.startswith('C|LIN') else 'C|HGB|ref' if n.startswith('C|HGB') else 'LIN|ref' if n.startswith('LIN')
            else 'RIDGE|escalado' if n.startswith('RIDGE') else 'HGB|ref' if n.startswith('HGB') else 'GBR|squared_error') for n in MEMORIA}

def veredicto_de(n):
    d = RES[n].rmse.to_numpy() - RES[REF_DE[n]].rmse.to_numpy()
    m, s = d.mean(), d.std(ddof=1)
    if n == REF_DE[n]: return '(referencia)'
    if abs(m) < 1e-6: return 'IDÉNTICO'
    if m < 0 and abs(m) > s: return 'MEJORA'
    if (m > 0 and m > s) or (d >= 0).all(): return 'EMPEORA'
    return 'EMPATE'

for n, (rm, sd, dm, ds, fm, le, gt, pc, ma, r2, ver) in MEMORIA.items():
    d = RES[n]; dif = d.rmse.to_numpy() - RES[REF_DE[n]].rmse.to_numpy()
    amplif = n.startswith('LIN|log')          # el log del lineal amplifica los errores numéricos: tolerancia mayor
    t1, t2 = (1.0, 2.0) if amplif else (0.05, 0.05)
    check('2.2 Variantes', f'{n}: RMSE medio (€)', rm, d.rmse.mean(), tol=t1)
    check('2.2 Variantes', f'{n}: std del RMSE', sd, d.rmse.std(ddof=1), tol=max(t1, 0.1))
    check('2.2 Variantes', f'{n}: dif. media frente a ref (€)', dm, dif.mean(), tol=t2)
    check('2.2 Variantes', f'{n}: folds mejores que la ref', fm, int((dif < 0).sum()), dec=0, tol=1)
    check('2.2 Variantes', f'{n}: RMSE <= 300 €', le, d.rmse_le300.mean(), tol=t1)
    check('2.2 Variantes', f'{n}: RMSE > 300 €', gt, d.rmse_gt300.mean(), tol=max(t1, 0.1))
    check('2.2 Variantes', f'{n}: % del MSE por encima de 300 €', pc, 100 * d.share_gt300.mean(), dec=1, tol=0.15)
    check('2.2 Variantes', f'{n}: MAE (€)', ma, d.mae.mean(), tol=t1)
    check('2.2 Variantes', f'{n}: R2', r2, d.r2.mean(), dec=3, tol=0.002)
    ver_c = veredicto_de(n)
    check_cond('2.2 Veredictos', f'{n}: veredicto', ver, ver_c == ver, ver_c)

# Esquema de validación
for modelo, rm, sd in [('LIN', 54.063, 5.034), ('HGB', 54.625, 4.591)]:
    v = RES[f'ESQ-aleatorio|{modelo}'].rmse
    check('2.1 Esquema', f'aleatorio {modelo}: RMSE medio', rm, v.mean(), tol=0.05)
    check('2.1 Esquema', f'aleatorio {modelo}: std del RMSE entre folds', sd, v.std(ddof=1), tol=0.05)
dd = RES['ESQ-aleatorio|LIN'].rmse.to_numpy() - RES['ESQ-aleatorio|HGB'].rmse.to_numpy()
check('2.1 Esquema', 'aleatorio: dif. lineal - HGB (media)', -0.562, dd.mean(), tol=0.05)
check('2.1 Esquema', 'aleatorio: dif. lineal - HGB (std)', 0.939, dd.std(ddof=1), tol=0.05)
for modelo, rm, sd in [('LIN', 54.174, 3.541), ('HGB', 54.455, 3.516)]:
    v = RES[f'ESQ-estratificado|{modelo}'].rmse
    check('2.1 Esquema', f'estratificado {modelo}: RMSE medio', rm, v.mean(), tol=0.05)
    check('2.1 Esquema', f'estratificado {modelo}: std del RMSE entre folds', sd, v.std(ddof=1), tol=0.05)
dd = RES['ESQ-estratificado|LIN'].rmse.to_numpy() - RES['ESQ-estratificado|HGB'].rmse.to_numpy()
check('2.1 Esquema', 'estratificado: dif. lineal - HGB (media)', -0.281, dd.mean(), tol=0.05)
check('2.1 Esquema', 'estratificado: dif. lineal - HGB (std)', 0.917, dd.std(ddof=1), tol=0.05)
check('2.1 Esquema', 'productos > 300 € por fold, mínimo (aleatorio)', 50, bal.aleatorio.min(), dec=0, tol=0)
check('2.1 Esquema', 'productos > 300 € por fold, máximo (aleatorio)', 97, bal.aleatorio.max(), dec=0, tol=0)
check('2.1 Esquema', 'productos > 300 € por fold, mínimo (estratificado)', 76, bal.estratificado.min(), dec=0, tol=0)
check('2.1 Esquema', 'productos > 300 € por fold, máximo (estratificado)', 80, bal.estratificado.max(), dec=0, tol=0)
check('2.1 Esquema', 'reducción de la std del RMSE lineal con estratificado (%)', 29.7, 100 * red['std RMSE lineal'], dec=1, tol=0.5)
check('2.1 Esquema', 'reducción de la std del RMSE HGB con estratificado (%)', 23.4, 100 * red['std RMSE HGB'], dec=1, tol=0.5)
check('2.1 Esquema', 'reducción de la std de la diferencia lineal-HGB con estratificado (%)', 2.3, 100 * red['std diferencia lineal-HGB'], dec=1, tol=0.5)
check_cond('2.1 Esquema', 'esquema elegido por la regla fijada de antemano', 'aleatorio', ESQUEMA == 'aleatorio', ESQUEMA)
check_cond('2.1 Esquema', 'particiones congeladas idénticas a las del autor (huella)', '0589e8baabcf', firma(SPLITS) == '0589e8baabcf', firma(SPLITS))
check('2.1 Esquema', 'nº de folds', 15, len(SPLITS), dec=0, tol=0)

# Diagnóstico del log en el lineal y redundancias exactas
check('2.3 Diagnóstico', 'folds del log lineal con RMSE > 100 €', 6, (dbg.RMSE > 100).sum(), dec=0, tol=0)
check('2.3 Diagnóstico', 'predicción máxima del log lineal en un fold (€)', 7948.2, dbg.pred_max.max(), dec=1, tol=1.0)
check('2.3 Diagnóstico', 'predicción máxima mínima entre los folds con RMSE > 100 € (€)', 4965.9, dbg.loc[dbg.RMSE > 100, 'pred_max'].min(), dec=1, tol=1.0)
check('2.3 Diagnóstico', 'peso mínimo del producto problemático en los folds con RMSE > 100 € (kg)', 18.1, dbg.loc[dbg.RMSE > 100, 'peso_kg'].min(), dec=1, tol=0.1)
check('2.3 Diagnóstico', 'productos con weight_kg > 10 en train', 9, (X.weight_kg > 10).sum(), dec=0, tol=0)
check('2.3 Diagnóstico', 'productos con weight_kg > 10 en test', 1, (X_test.weight_kg > 10).sum(), dec=0, tol=0)
check('2.3 Diagnóstico', 'peso máximo en test (kg)', 12.783, X_test.weight_kg.max(), dec=3, tol=0.0005)
check_cond('2.3 Redundancias', 'lineal sin brand_tier = lineal con brand_tier (RMSE por fold idéntico)', 'idéntico',
           np.abs(RES['LIN|sin brand_tier'].rmse - RES['LIN|ref'].rmse).max() < 1e-6, np.abs(RES['LIN|sin brand_tier'].rmse - RES['LIN|ref'].rmse).max())
check_cond('2.3 Redundancias', 'lineal por mínimos cuadrados: escalar no cambia el RMSE por fold', 'idéntico',
           np.abs(RES['LIN|sin escalar'].rmse - RES['LIN|ref'].rmse).max() < 1e-6, np.abs(RES['LIN|sin escalar'].rmse - RES['LIN|ref'].rmse).max())
check_cond('2.3 Datos', 'id no es predictora', 'ausente', ('id' not in X.columns) and ('id' not in X_test.columns), ','.join(X.columns))

res_chk = pd.DataFrame(CHECKS, columns=['sección', 'afirmación', 'memoria', 'calculado', 'estado'])
res2 = res_chk[res_chk['sección'].str.startswith('2.')]
print(res2.groupby('sección').estado.value_counts().unstack(fill_value=0))
print(f"\nParte 2: {(res2.estado == 'OK').sum()} OK de {len(res2)} comprobaciones")
if (res2.estado != 'OK').any():
    print('\nFILAS A REVISAR (probable diferencia de versiones de librerías):')
    print(res2[res2.estado != 'OK'].to_string(index=False))
else:
    print('Todas las cifras de la memoria de la Fase 2 coinciden con lo calculado aquí.')

## Qué queda congelado para la fase 3
- Esquema de validación: KFold aleatorio, 5 folds x 3 repeticiones, `random_state=42` (`SPLITS`). CV de control: semilla 2026 (`SPLITS_CTRL`).
- Funciones reutilizables: `build(model, target, **opciones)`, `evaluar`, `resumen`, `mostrar`.
- Conjunto de variables de trabajo: `drop=SIN` (sin `material`, `brand_tier`, `is_promo`, `country`). La fase 5 volverá a comparar con el conjunto completo con los hiperparámetros ya ajustados.
- Candidatos abiertos: objetivo `log_duan` en los modelos de árboles; categóricas nativas en HGB.